## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [96]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [97]:
# For installing the libraries & downloading models from HF Hub
# !pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.0 numpy==2.3.3 -q

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [98]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

#Pretty print helper
import pprint

def output(text):
  pprint.pprint(text, width=120)

## Question Answering using LLM

#### Downloading and Loading the model

In [99]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

llm = Llama(
    model_path=model_path,
    n_ctx=5000,
    n_gpu_layers=38,
    n_batch=512
)

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /Users/amit/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q6_K.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model

In [100]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    output( ">>>>>>>>>Calling llm with mt=" + str(max_tokens) + " temp=" + str(temperature) + " tp=" + str(top_p) + " tk=" + str(top_k) )
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [101]:
q1 = "What is the protocol for managing sepsis in a critical care unit?"
response(q1,max_tokens=128,temperature=0,top_p=0.95,top_k=50)


'>>>>>>>>>Calling llm with mt=128 temp=0 tp=0.95 tk=50'



llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      14.38 ms /   128 runs   (    0.11 ms per token,  8904.35 tokens per second)
llama_print_timings: prompt eval time =    3479.99 ms /    16 tokens (  217.50 ms per token,     4.60 tokens per second)
llama_print_timings:        eval time =    6582.45 ms /   127 runs   (   51.83 ms per token,    19.29 tokens per second)
llama_print_timings:       total time =   10323.89 ms


'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. Source control'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [102]:
q2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(q2,max_tokens=128,temperature=0,top_p=0.95,top_k=50)

Llama.generate: prefix-match hit


'>>>>>>>>>Calling llm with mt=128 temp=0 tp=0.95 tk=50'



llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      13.58 ms /   128 runs   (    0.11 ms per token,  9427.71 tokens per second)
llama_print_timings: prompt eval time =     137.77 ms /    32 tokens (    4.31 ms per token,   232.27 tokens per second)
llama_print_timings:        eval time =    6360.86 ms /   127 runs   (   50.09 ms per token,    19.97 tokens per second)
llama_print_timings:       total time =    6754.28 ms


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The symptoms of appendicitis can vary from person to person, but some common signs include:\n\n1. Abdominal pain: The pain is typically located in the lower right quadrant of the abdomen and may start as a mild discomfort that gradually worsens over time. The pain may be constant or intermittent and may be aggravated by movement, deep breathing, or coughing.\n2. Loss of appetite'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [103]:
q3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(q3,max_tokens=128,temperature=0,top_p=0.95,top_k=50)

Llama.generate: prefix-match hit


'>>>>>>>>>Calling llm with mt=128 temp=0 tp=0.95 tk=50'



llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      13.46 ms /   128 runs   (    0.11 ms per token,  9512.49 tokens per second)
llama_print_timings: prompt eval time =     237.12 ms /    34 tokens (    6.97 ms per token,   143.38 tokens per second)
llama_print_timings:        eval time =    6436.99 ms /   127 runs   (   50.69 ms per token,    19.73 tokens per second)
llama_print_timings:       total time =    6926.44 ms


"\n\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, or eyelashes.\n\nThe exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications."

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [104]:
q4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(q4,max_tokens=128,temperature=0,top_p=0.95,top_k=50)

Llama.generate: prefix-match hit


'>>>>>>>>>Calling llm with mt=128 temp=0 tp=0.95 tk=50'



llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      13.33 ms /   128 runs   (    0.10 ms per token,  9603.84 tokens per second)
llama_print_timings: prompt eval time =     140.73 ms /    28 tokens (    5.03 ms per token,   198.96 tokens per second)
llama_print_timings:        eval time =    6556.69 ms /   127 runs   (   51.63 ms per token,    19.37 tokens per second)
llama_print_timings:       total time =    6958.15 ms


'\n\nA person who has sustained a physical injury to the brain tissue may require various treatments depending on the severity and location of the injury. Here are some common treatments that may be recommended:\n\n1. Emergency care: In case of a traumatic brain injury (TBI), it is essential to seek emergency medical attention as soon as possible. The primary goal of emergency care is to prevent further damage to the brain, stabilize vital signs, and manage any life-threatening conditions.\n2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage various conditions associated with a'

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [105]:
q5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(q5,max_tokens=128,temperature=0,top_p=0.95,top_k=50)

Llama.generate: prefix-match hit


'>>>>>>>>>Calling llm with mt=128 temp=0 tp=0.95 tk=50'



llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      12.68 ms /   128 runs   (    0.10 ms per token, 10098.62 tokens per second)
llama_print_timings: prompt eval time =     249.35 ms /    35 tokens (    7.12 ms per token,   140.36 tokens per second)
llama_print_timings:        eval time =    6343.69 ms /   127 runs   (   49.95 ms per token,    20.02 tokens per second)
llama_print_timings:       total time =    6815.17 ms


"\n\nFirst and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:\n\n1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.\n2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.\n3. Immobilize the leg: Use a splint, sl"

**Observations:**


* When prompting the LLM without any context and using the default parameters (max_tokens, temperature, top_p, top_k) the responses are very limited in that they are vague, short, and sometimes truncated.
* The response time is very quick, less than 10 seconds.

## Question Answering using LLM with Prompt Engineering

In [106]:
llm_query_context = """
You are an expert whose work is to review the report and provide the appropriate answers from the context.
If the answer is not found in the context, respond "I don't know".
"""

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [107]:
#Five different LLM parameter combinations used in this notebook
paramList = data = [
    {"max_tokens": 1000, "temp": 0, "top_p" : 0.95, "top_k":  50},
    {"max_tokens": 1000, "temp": 1, "top_p" : 0.85, "top_k":  100},
    {"max_tokens": 1000, "temp": 2, "top_p" : 0.75, "top_k":  250},
    {"max_tokens": 1000, "temp": 3, "top_p" : 0.65, "top_k":  500},
    {"max_tokens": 1000, "temp": 4, "top_p" : 0.45, "top_k":  500},
]

In [108]:
for params in paramList:
  output(response(llm_query_context + q1,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

Llama.generate: prefix-match hit


'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'



llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      41.12 ms /   381 runs   (    0.11 ms per token,  9266.69 tokens per second)
llama_print_timings: prompt eval time =     244.27 ms /    57 tokens (    4.29 ms per token,   233.35 tokens per second)
llama_print_timings:        eval time =   18768.57 ms /   380 runs   (   49.39 ms per token,    20.25 tokens per second)
llama_print_timings:       total time =   19828.24 ms
Llama.generate: prefix-match hit


('\n'
 'Based on the context provided, the protocol for managing sepsis in a critical care unit may include the following '
 'steps:\n'
 '1. Early recognition and diagnosis of sepsis using clinical criteria such as the Sequential Organ Failure Assessment '
 '(SOFA) score or Quick Sequential Organ Failure Assessment (qSOFA) score.\n'
 '2. Immediate initiation of antibiotic therapy based on culture results, if available, or empiric therapy based on '
 "the patient's clinical presentation and local guidelines.\n"
 '3. Fluid resuscitation to maintain adequate tissue perfusion and organ function, with careful monitoring of fluid '
 'balance and hemodynamic status.\n'
 '4. Administration of vasopressors or inotropes as needed to maintain mean arterial pressure (MAP) above 65 mmHg.\n'
 '5. Close monitoring of vital signs, laboratory values, and organ function, including urine output, respiratory '
 'status, and cardiac output.\n'
 '6. Use of adjunctive therapies such as corticosteroids, vasoa


llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      36.28 ms /   372 runs   (    0.10 ms per token, 10254.43 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   18488.54 ms /   372 runs   (   49.70 ms per token,    20.12 tokens per second)
llama_print_timings:       total time =   19200.97 ms
Llama.generate: prefix-match hit


...


Llama.generate: prefix-match hit
llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      58.87 ms /   465 runs   (    0.13 ms per token,  7898.22 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   22964.12 ms /   465 runs   (   49.39 ms per token,    20.25 tokens per second)
llama_print_timings:       total time =   23870.70 ms


('\n'
 'The protocol for managing sepsis in a critical care unit typically involves the following steps:\n'
 '1. Early recognition and suspicion of sepsis based on clinical signs and laboratory results.\n'
 '2. Immediate initiation of antibiotic therapy based on culture results or suspected infection site.\n'
 '3. Administration of intravenous fluids to maintain adequate blood pressure and organ perfusion.\n'
 '4. Monitoring of vital signs, urine output, and lactate levels.\n'
 '5. Use of vasopressors if necessary to maintain adequate blood pressure.\n'
 '6. Close monitoring for organ dysfunction or failure and appropriate interventions as needed.\n'
 '7. Addressing any underlying causes of sepsis, such as source control for infections.\n'
 '8. Multidisciplinary involvement, including consultation with infectious disease specialists, intensive care unit '
 'teams, and other relevant specialists.\n'
 '9. Providing supportive care, such as mechanical ventilation or renal replacement ther


llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      43.62 ms /   254 runs   (    0.17 ms per token,  5823.02 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   12475.15 ms /   254 runs   (   49.11 ms per token,    20.36 tokens per second)
llama_print_timings:       total time =   12968.50 ms
Llama.generate: prefix-match hit


('\n'
 "Based on the context provided, there isn't enough information to determine the specific protocol for managing sepsis "
 'in a critical care unit. However, generally speaking, the management of sepsis in a critical care unit involves '
 'early recognition and prompt initiation of appropriate antibiotic therapy, fluid resuscitation, and supportive '
 'measures such as mechanical ventilation and vasopressor support if necessary. Close monitoring of vital signs, '
 'laboratory values, and organ function is also essential. Additionally, source control, such as drainage or surgical '
 'intervention, may be required depending on the underlying cause of sepsis.')
'                                      ---------------------------'



llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      21.58 ms /   125 runs   (    0.17 ms per token,  5793.21 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =    6053.22 ms /   125 runs   (   48.43 ms per token,    20.65 tokens per second)
llama_print_timings:       total time =    6289.02 ms


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [109]:
for params in paramList:
  output(response(llm_query_context + q2,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

Llama.generate: prefix-match hit


'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'



llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      27.92 ms /   283 runs   (    0.10 ms per token, 10135.01 tokens per second)
llama_print_timings: prompt eval time =     139.50 ms /    32 tokens (    4.36 ms per token,   229.38 tokens per second)
llama_print_timings:        eval time =   13935.00 ms /   282 runs   (   49.41 ms per token,    20.24 tokens per second)
llama_print_timings:       total time =   14606.33 ms
Llama.generate: prefix-match hit


('\n'
 'According to the Mayo Clinic, the common symptoms of appendicitis include:\n'
 '1. Sudden loss of appetite\n'
 '2. Pain in your lower right-hand side of your abdomen (belly), particularly around your navel and sometimes shifting '
 'to your lower right side\n'
 '3. Feeling sick and vomiting\n'
 '4. Fever, which may start a few hours after the stomach pain begins\n'
 '5. Diarrhea or constipation\n'
 '6. Swelling in the area of the appendix, which is located near the navel and the upper right part of your abdomen\n'
 '7. Inability to pass gas or have a bowel movement\n'
 '8. Pain that worsens when you walk, cough, deep breathe or move suddenly\n'
 '9. Loss of energy and general malaise\n'
 'Appendicitis cannot be cured via medicine alone. If the appendix becomes ruptured or bursts, it can lead to '
 'peritonitis, a serious inflammation of the abdominal cavity that requires immediate surgery. The standard surgical '
 'procedure for treating appendicitis is an appendectomy, which i


llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      18.17 ms /   169 runs   (    0.11 ms per token,  9303.09 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =    8352.62 ms /   169 runs   (   49.42 ms per token,    20.23 tokens per second)
llama_print_timings:       total time =    8698.28 ms
Llama.generate: prefix-match hit


('\n'
 'The common symptoms of appendicitis include:\n'
 '1. Sudden onset of pain in the lower right abdomen\n'
 '2. Loss of appetite\n'
 '3. Nausea and vomiting\n'
 '4. Fever\n'
 '5. Constipation or diarrhea\n'
 '6. Abdominal swelling\n'
 '7. Inability to pass gas or have a bowel movement\n'
 '8. Pain in the lower back, especially when lying down\n'
 'Appendicitis cannot be cured via medicine alone. If the diagnosis is confirmed through various tests such as '
 'ultrasound or CT scan, then surgery is the only treatment option. The surgical procedure for appendicitis is called '
 'an appendectomy, which involves removing the inflamed appendix from the body to prevent it from rupturing and '
 'causing further complications.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=2 tp=0.75 tk=250'
('\n'
 'Based on the provided context, the common symptoms for appendicitis include abdominal pain, usually located in the '
 'lower right 


llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      16.65 ms /   130 runs   (    0.13 ms per token,  7807.34 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =    6330.57 ms /   130 runs   (   48.70 ms per token,    20.54 tokens per second)
llama_print_timings:       total time =    6573.85 ms
Llama.generate: prefix-match hit

llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      24.11 ms /   146 runs   (    0.17 ms per token,  6054.57 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =    7122.95 ms /   146 runs   (   48.79 ms per token,    20.50 tokens per second)
llama_print_timings:       total time =    7379.93 ms
Llama.generate: prefix-

('\n'
 'Based on the context provided in the report, the common symptoms for appendicitis include abdominal pain, loss of '
 'appetite, nausea, vomiting, fever, and a tender mass in the lower right quadrant of the abdomen. The report also '
 'states that appendicitis cannot be cured via medicine alone and requires surgical intervention. Therefore, the '
 'appropriate answer is:\n'
 'The common symptoms for appendicitis include abdominal pain, loss of appetite, nausea, vomiting, fever, and a tender '
 'mass in the lower right quadrant of the ababelly. Appendicitis cannot be cured via medicine alone and requires '
 'surgical intervention, typically an appendectomy.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=4 tp=0.45 tk=500'
('\n'
 'According to the Mayo Clinic, the common symptoms of appendicitis include:\n'
 '1. Sudden loss of appetite\n'
 '2. Pain in your lower right-hand side of your abdomen (around your navel area) t


llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      51.57 ms /   310 runs   (    0.17 ms per token,  6011.36 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   15155.28 ms /   310 runs   (   48.89 ms per token,    20.45 tokens per second)
llama_print_timings:       total time =   15740.81 ms


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [110]:
for params in paramList:
  output(response(llm_query_context + q3,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

Llama.generate: prefix-match hit


'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'



llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      44.90 ms /   467 runs   (    0.10 ms per token, 10400.66 tokens per second)
llama_print_timings: prompt eval time =     243.83 ms /    34 tokens (    7.17 ms per token,   139.44 tokens per second)
llama_print_timings:        eval time =   22923.15 ms /   466 runs   (   49.19 ms per token,    20.33 tokens per second)
llama_print_timings:       total time =   24037.27 ms
Llama.generate: prefix-match hit


('\n'
 'According to the American Academy of Dermatology (AAD), there are several potential causes for sudden, patchy hair '
 'loss, including:\n'
 '1. Alopecia Areata: This is an autoimmune disorder that causes hair loss in circular patches on the scalp, face, or '
 'other areas of the body. It can affect people of all ages and ethnicities. There is no cure for alopecia areata, but '
 'various treatments can help stimulate hair regrowth. These include:\n'
 '   a. Corticosteroids: Topical or injected steroids can reduce inflammation and promote hair growth.\n'
 '   b. Immunotherapy: Injections of certain substances can help the immune system to attack the cells that cause hair '
 'loss.\n'
 '   c. Minoxidil: A topical medication that can stimulate hair growth in some people.\n'
 '2. Telogen Effluvium: This is a condition where the hair enters the resting phase prematurely, leading to shedding. '
 'It can be caused by stress, illness, or certain medications. Treatment for telogen effluv


llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      61.60 ms /   637 runs   (    0.10 ms per token, 10340.24 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   31443.57 ms /   637 runs   (   49.36 ms per token,    20.26 tokens per second)
llama_print_timings:       total time =   32699.17 ms
Llama.generate: prefix-match hit


...


llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      71.58 ms /   589 runs   (    0.12 ms per token,  8229.13 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   28262.55 ms /   589 runs   (   47.98 ms per token,    20.84 tokens per second)
llama_print_timings:       total time =   29400.74 ms
Llama.generate: prefix-match hit


('\n'
 'From the context provided, I cannot determine the specific effective treatments or solutions for addressing sudden '
 'patchy hair loss. However, some common approaches include:\n'
 '1. Topical treatments such as minoxidil (Rogaine) or corticosteroids.\n'
 '2. Oral medications like finasteride (Propecia).\n'
 '3. Platelet-rich plasma (PRP) therapy.\n'
 '4. Hair transplantation.\n'
 '5. Lifestyle modifications, such as reducing stress and improving diet.\n'
 'As for the possible causes behind sudden patchy hair loss, they can include:\n'
 '1. Alopecia areata, an autoimmune disorder that causes hair loss in round patches.\n'
 '2. Trauma or injury to the scalp.\n'
 '3. Hormonal imbalances, such as thyroid issues or polycystic ovary syndrome (PCOS).\n'
 '4. Nutritional deficiencies.\n'
 '5. Medications like chemotherapy drugs, antidepressants, and blood thinners.\n'
 '6. Stress or anxiety.\n'
 '7. Fungal infections.\n'
 '8. Trichotillomania, a hair-pulling disorder.\n'
 'It is impo


llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      46.65 ms /   271 runs   (    0.17 ms per token,  5809.72 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   13338.54 ms /   271 runs   (   49.22 ms per token,    20.32 tokens per second)
llama_print_timings:       total time =   13871.07 ms
Llama.generate: prefix-match hit


('\n'
 'According to the American Academy of Dermatology (AAD), there are several potential causes for sudden, patchy hair '
 'loss, including:\n'
 '1. Alopecia Areata: This is an autoimmune disorder that causes hair loss in small, round patches on the scalp, face, '
 'or other areas of the body. It often results in smooth, bald spots.\n'
 '2. Telogen Effluvium: This condition occurs when a large number of hairs enter the resting phase (telogen) at the '
 'same time, causing shedding and thinning. It can be caused by stress, illness, or hormonal changes.\n'
 '3. Traction Alopecia: This is a type of hair loss that occurs when the hair is pulled too tightly, leading to damage '
 'and eventual hair loss. It often affects the edges of the hairline or the crown of the head.\n'
 '4. Scarring Alopecia: This type of hair loss occurs when the hair follicles are destroyed, leading to permanent '
 'baldness. It can be caused by injuries, burns, or certain medical conditions.\n'
 'As for effective


llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =     100.91 ms /   606 runs   (    0.17 ms per token,  6005.29 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   28768.90 ms /   606 runs   (   47.47 ms per token,    21.06 tokens per second)
llama_print_timings:       total time =   29965.96 ms


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [111]:
for params in paramList:
  output(response(llm_query_context + q4,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

Llama.generate: prefix-match hit


'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'



llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      18.29 ms /   173 runs   (    0.11 ms per token,  9460.27 tokens per second)
llama_print_timings: prompt eval time =     139.39 ms /    28 tokens (    4.98 ms per token,   200.87 tokens per second)
llama_print_timings:        eval time =    8034.77 ms /   172 runs   (   46.71 ms per token,    21.41 tokens per second)
llama_print_timings:       total time =    8472.37 ms
Llama.generate: prefix-match hit


('\n'
 'Based on the context provided, there is no specific information about the recommended treatments for a person with a '
 'physical injury to brain tissue. However, some common treatments for brain injuries include:\n'
 '1. Medications to manage symptoms such as pain, swelling, and seizures.\n'
 '2. Rehabilitation therapy, including physical therapy, occupational therapy, speech therapy, and cognitive '
 'rehabilitation.\n'
 '3. Surgery to remove hematomas or other obstructions.\n'
 '4. Assistive devices, such as braces or prosthetics, to help with mobility and function.\n'
 '5. Psychological counseling to address emotional and behavioral changes.\n'
 'It is important to note that the specific treatment plan will depend on the severity and location of the injury, as '
 "well as the individual's overall health and response to treatment.")
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=1 tp=0.85 tk=100'
('\n'
 'Based on t


llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =       8.32 ms /    85 runs   (    0.10 ms per token, 10211.44 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =    3972.33 ms /    85 runs   (   46.73 ms per token,    21.40 tokens per second)
llama_print_timings:       total time =    4113.96 ms
Llama.generate: prefix-match hit

llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      45.92 ms /   376 runs   (    0.12 ms per token,  8187.98 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   17930.28 ms /   376 runs   (   47.69 ms per token,    20.97 tokens per second)
llama_print_timings:       total time =   18620.77 ms
Llama.generate: prefix-

('\n'
 '\n'
 'According to the Mayo Clinic, treatment for a brain injury may include:\n'
 '1. Emergency medical care: This is essential for stabilizing the patient and addressing any life-threatening '
 'conditions, such as bleeding, swelling, or infections.\n'
 '2. Medications: Depending on the type and severity of the injury, medications may be prescribed to manage symptoms '
 'like pain, seizures, or infections.\n'
 '3. Rehabilitation therapy: This can include physical therapy, occupational therapy, speech therapy, and cognitive '
 'rehabilitation to help the person regain lost functions and improve overall quality of life.\n'
 '4. Assistive devices: Devices such as wheelchairs, braces, or communication aids may be necessary to help the person '
 'perform daily activities and manage any impairments.\n'
 '5. Surgery: In some cases, surgery may be required to remove blood clots, repair skull fractures, or relieve '
 'pressure on the brain.\n'
 "6. Lifestyle modifications: This may inc


llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      32.07 ms /   188 runs   (    0.17 ms per token,  5862.91 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =    8773.67 ms /   188 runs   (   46.67 ms per token,    21.43 tokens per second)
llama_print_timings:       total time =    9112.67 ms
Llama.generate: prefix-match hit


('\n'
 'According to the Mayo Clinic, treatment for a brain injury can include:\n'
 '1. Medications to manage symptoms such as pain, seizures, and swelling in the brain.\n'
 '2. Surgery to remove blood clots or repair skull fractures.\n'
 '3. Rehabilitation therapy, including physical therapy, occupational therapy, speech-language therapy, and cognitive '
 'rehabilitation, to help the person regain lost skills and functions.\n'
 '4. Assistive devices such as wheelchairs, braces, or communication aids to help the person function independently.\n'
 '5. Psychological counseling to help the person cope with emotional and behavioral changes.\n'
 "I don't know if the context specifies whether the impairment is temporary or permanent, but based on the information "
 'provided by the Mayo Clinic, both types of injuries would likely require some combination of these treatments.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=4 tp=0.4


llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      56.84 ms /   347 runs   (    0.16 ms per token,  6105.39 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   16317.46 ms /   347 runs   (   47.02 ms per token,    21.27 tokens per second)
llama_print_timings:       total time =   16953.15 ms


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [112]:
for params in paramList:
  output(response(llm_query_context + q5,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

Llama.generate: prefix-match hit


'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'
('\n'
 'According to Mayo Clinic, if you suspect that you or someone else has fractured a leg during a hiking trip, follow '
 'these steps:\n'
 '1. Stay calm and try to keep the person as still as possible to prevent further injury.\n'
 '2. Apply a splint or immobilize the leg with a makeshift sling or other material if possible.\n'
 '3. If the fracture is open (compound), try to clean the wound gently with water and a sterile gauze, but do not '
 'attempt to close it yourself.\n'
 '4. Do not move the person unless they are in danger or if necessary to prevent further injury.\n'
 '5. Seek medical help as soon as possible.\n'
 'Regarding treatment steps for a fractured leg, according to Mayo Clinic, the following procedures may be necessary:\n'
 '1. Immobilization with a cast or brace to keep the bone in place and allow it to heal.\n'
 '2. Surgery to realign the bone if it is severely displaced or if there is damage to the soft ti


llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      37.44 ms /   370 runs   (    0.10 ms per token,  9883.27 tokens per second)
llama_print_timings: prompt eval time =     236.55 ms /    35 tokens (    6.76 ms per token,   147.96 tokens per second)
llama_print_timings:        eval time =   17390.10 ms /   369 runs   (   47.13 ms per token,    21.22 tokens per second)
llama_print_timings:       total time =   18321.10 ms
Llama.generate: prefix-match hit

llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      45.84 ms /   454 runs   (    0.10 ms per token,  9902.93 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   21495.40 ms /   454 runs   (   47.35 ms per token,    21.12 tokens per second)
llama_print_timings:       total time =   22351.05 ms
Llama.generate: prefix-

('\n'
 'According to the American Academy of Orthopaedic Surgeons (AAOS), if you suspect that you or someone else has '
 'fractured a leg during a hiking trip, follow these steps:\n'
 '1. Stay calm and try to immobilize the injured leg with a splint or a makeshift sling to prevent any further '
 'damage.\n'
 '2. Apply ice packs to reduce swelling and pain. Do not apply ice directly to the skin; wrap it in a cloth first.\n'
 '3. Elevate the injured leg above heart level to minimize swelling and bruising.\n'
 '4. Seek medical attention as soon as possible. A fracture cannot be healed without proper medical treatment, and '
 'delaying treatment can lead to complications such as infection or improper healing.\n'
 '5. In the meantime, provide the person with fluids to prevent dehydration and maintain body temperature.\n'
 '6. Monitor vital signs such as pulse, breathing rate, and blood pressure.\n'
 '7. Keep the person comfortable and calm to reduce anxiety and stress.\n'
 '8. If the person


llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      48.72 ms /   397 runs   (    0.12 ms per token,  8148.10 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   18708.15 ms /   397 runs   (   47.12 ms per token,    21.22 tokens per second)
llama_print_timings:       total time =   19428.34 ms
Llama.generate: prefix-match hit


('\n'
 'Based on the provided context, the following information can be gathered:\n'
 '1. The person has sustained a fracture in their leg while hiking.\n'
 '2. It is important to immobilize the leg to prevent further damage and promote healing.\n'
 '3. The person should be transported to a medical facility as soon as possible for proper diagnosis and treatment.\n'
 '4. Depending on the severity of the fracture, surgery may be required.\n'
 '5. Once the fracture has healed, rehabilitation exercises will be necessary to regain strength and mobility in the '
 'leg.\n'
 '6. Proper care and attention during the recovery process are essential to prevent complications and ensure a full '
 'and proper healing of the fracture.\n'
 'Therefore, the necessary precautions for a person who has fractured their leg during a hiking trip include '
 'immobilizing the leg to prevent further damage and promote healing, transporting them to a medical facility as soon '
 'as possible for proper diagnosis an


llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      68.78 ms /   417 runs   (    0.16 ms per token,  6062.54 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   19603.06 ms /   417 runs   (   47.01 ms per token,    21.27 tokens per second)
llama_print_timings:       total time =   20383.28 ms
Llama.generate: prefix-match hit


('\n'
 'Based on the context provided, there is no specific information given about the necessary precautions and treatment '
 'steps for a person who has fractured their leg during a hiking trip. However, I can provide some general guidelines '
 'based on common medical advice.\n'
 '1. Assess the severity of the injury: If the fracture is open (compound), there may be an increased risk of '
 "infection, so it's important to assess the wound for signs of infection such as redness, swelling, or drainage. If "
 'the fracture is closed (simple), there may still be swelling and pain.\n'
 '2. Immobilize the leg: Use a splint or sling to immobilize the leg and prevent any further movement that could '
 'worsen the injury. This will also help reduce pain and swelling.\n'
 "3. Seek medical attention: It's important to seek medical attention as soon as possible, even if it means waiting "
 'for rescue services to reach you. A fracture can cause complications such as blood clots or nerve damage 


llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      60.04 ms /   353 runs   (    0.17 ms per token,  5879.41 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   16687.12 ms /   353 runs   (   47.27 ms per token,    21.15 tokens per second)
llama_print_timings:       total time =   17356.13 ms


In [113]:
paramList

[{'max_tokens': 1000, 'temp': 0, 'top_p': 0.95, 'top_k': 50},
 {'max_tokens': 1000, 'temp': 1, 'top_p': 0.85, 'top_k': 100},
 {'max_tokens': 1000, 'temp': 2, 'top_p': 0.75, 'top_k': 250},
 {'max_tokens': 1000, 'temp': 3, 'top_p': 0.65, 'top_k': 500},
 {'max_tokens': 1000, 'temp': 4, 'top_p': 0.45, 'top_k': 500}]

*   As parameters are changed over five iterations, the response gets more wordy while conveying much of the same information.  It also provides more details about what should be done post medical care, such as avoiding certain activities and getting rehabilitation care.  
*   The third of five parameter sets provides a good balance between wordiness and the amount of information provided.
* Providing context makes the queries run longer, on the order 20-30 seconds each.



## Data Preparation for RAG

### Loading the Data

In [114]:
#from google.colab import drive
# drive.mount('/content/drive')


In [115]:
#Load the data file provided
pdf_path = "medical_diagnosis_manual.pdf"
pdf_loader = PyMuPDFLoader(pdf_path)
manual = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [116]:
#Verify loading of pdf file
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")

Page Number : 1
apgt60@hotmail.com
ICXHW2DZL4
This file is meant for personal use by apgt60@hotmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
apgt60@hotmail.com
ICXHW2DZL4
This file is meant for personal use by apgt60@hotmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...........................................................................................................................................

#### Checking the number of pages

In [117]:
#Check length of loaded manual
len(manual)

4114

### Data Chunking

In [118]:
#Split the data using a text splitter with necessary attributes
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap= 50
)

document_chunks = pdf_loader.load_and_split(text_splitter)

In [119]:
#Verify length of chunks
len(document_chunks)

8659

In [120]:
#Verify content of chunks
document_chunks[0].page_content

'apgt60@hotmail.com\nICXHW2DZL4\nThis file is meant for personal use by apgt60@hotmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [121]:
#Verify content of chunks
document_chunks[-2].page_content

'Y\nYaws 1266-1267\nforest 1379\nY chromosome 3373 (see also Genetic)\nabnormalities of 3005\nYeast infection (see also Fungal infection)\nvaginal 2542, 2544, 2545\nYellow fever 1400, 1429, 1437\nhepatic inflammation in 248\nvaccine against 1172, 1437, 3441\nYellow nail syndrome 732, 1995\npleural effusion in 1997\nYellow skin (see Jaundice)\nYersinia infection 1167, 1256-1257\nY. enterocolitica infection 147\nY. pestis infection 1924\nYew poisoning 3338\nYips 1762\nYo, antibodies to 1056\nYolk sac tumor 2476\nThe Merck Manual of Diagnosis & Therapy, 19th Edition\nY\n4103\napgt60@hotmail.com\nICXHW2DZL4\nThis file is meant for personal use by apgt60@hotmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

### Embedding

In [122]:
#Pick an embedding model
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')


In [123]:
#Create two embeddings and verify the lengths
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

### Vector Database

In [124]:
out_dir = 'merck_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [125]:
#Load the vector db - long running process!
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [126]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

### Retriever

In [127]:
#Function to get a retreiver with search method and k value
def get_retreiver(search_type='similarity',k=2):
    return vectorstore.as_retriever(
      search_type=search_type,
      search_kwargs={'k': k}
)

In [128]:
#Run a query against the vector db
rel_docs = get_retreiver().get_relevant_documents(q1)
rel_docs

[Document(metadata={'author': '', 'subject': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'trapped': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'keywords': '', 'file_path': 'medical_diagnosis_manual.pdf', 'source': 'medical_diagnosis_manual.pdf', 'moddate': '2026-03-17T05:27:32+00:00', 'total_pages': 4114, 'page': 2455, 'format': 'PDF 1.7', 'creator': 'Atop CHM to PDF Converter', 'modDate': 'D:20260317052732Z', 'creationDate': 'D:20120615054440Z'}, page_content='• Normalization of blood glucose levels\n• Replacement-dose corticosteroids\nPatients with septic shock should be treated in an ICU. The following should be monitored frequently (see\nalso p. 2244): systemic pressure; CVP, PAOP, or both; pulse oximetry; ABGs; blood glucose, lactate, and\nelectrolyte levels; renal function, and possibly sublingual PCO2. Urine output, a good indicator of renal\nperfusion, should be measured, usually with an

In [129]:
#Check how many docs were returned
len(rel_docs)

2

### System and User Prompt Template

In [130]:
#1. A system message describing the expert's role.
#2. A user message template including context and the question.

system_message = """
You are an expert whose work is to review the manual and provide the appropriate answers from the context.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.
The context ends with the token: ###EndContext.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Do not mention anything about the context in your final answer.

If the answer is not found in the context, respond "I don't know".

"""


qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}
###EndContext

###Question
{question}
"""

### Response Function

In [131]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = get_retreiver(k=k).get_relevant_documents(query=user_input)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = system_message + '\n' + user_message

    output( ">>>>>>>>>Calling llm with mt=" + str(max_tokens) + " temp=" + str(temperature) + " tp=" + str(top_p) + " tk=" + str(top_k) )
    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [132]:
for params in paramList:
  output(generate_rag_response(q1,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'


Llama.generate: prefix-match hit

llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      40.29 ms /   368 runs   (    0.11 ms per token,  9133.10 tokens per second)
llama_print_timings: prompt eval time =   13766.25 ms /  1811 tokens (    7.60 ms per token,   131.55 tokens per second)
llama_print_timings:        eval time =   20584.15 ms /   367 runs   (   56.09 ms per token,    17.83 tokens per second)
llama_print_timings:       total time =   35248.79 ms


('Answer: The protocol for managing sepsis in a critical care unit includes monitoring systemic pressure, CVP, PAOP, '
 'pulse oximetry, ABGs, blood glucose, lactate and electrolyte levels, renal function, and possibly sublingual PCO2. '
 'Urine output should be measured using an indwelling catheter. Fluid resuscitation with 0.9% saline should be given '
 'until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg. Oliguria with hypotension is not a '
 'contraindication to vigorous fluid resuscitation, and the quantity of fluid required often exceeds the normal blood '
 'volume and may reach 10 L over 4 to 12 hours. PAOP or echocardiography can identify limitations in left ventricular '
 'function and incipient pulmonary edema due to fluid overload. If a patient remains hypotensive after CVP or PAOP has '
 'been raised to target levels, dopamine may be given to increase mean BP to at least 60 mm Hg. If dopamine dose '
 'exceeds 20 μg/kg/min, another vasopressor, typically nor

Llama.generate: prefix-match hit


('Answer: The protocol includes monitoring systemic pressure, CVP or PAOP, pulse oximetry, ABGs, blood glucose, '
 'lactate and electrolyte levels, renal function, and possibly sublingual PCO2. Urine output should be measured using '
 'an indwelling catheter. Fluid resuscitation with 0.9% saline should continue until CVP reaches 8 mm Hg (10 cm H2O) '
 'or PAOP reaches 12 to 15 mm Hg. If the patient remains hypotensive, dopamine may be administered to increase mean BP '
 'to at least 60 mm Hg. However, if the dopamine dose exceeds 20 μg/kg/min, another vasopressor like norepinephrine '
 'may be added, but this comes with risks of organ hypoperfusion and acidosis. Oxygen should be given by mask or nasal '
 'prongs and tracheal intubation and mechanical ventilation may be needed for respiratory failure. Normalization of '
 'blood glucose levels and replacement-dose corticosteroids are also part of the treatment plan.')
'                                      ---------------------------'


...


llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      21.51 ms /    55 runs   (    0.39 ms per token,  2556.83 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   11827.94 ms /    55 runs   (  215.05 ms per token,     4.65 tokens per second)
llama_print_timings:       total time =   12265.57 ms


'>>>>>>>>>Calling llm with mt=1000 temp=4 tp=0.45 tk=500'


Llama.generate: prefix-match hit


('Answer: The protocol for managing sepsis in a critical care unit includes monitoring systemic pressure, CVP, PAOP, '
 'pulse oximetry, ABGs, blood glucose, lactate and electrolyte levels, renal function, and possibly sublingual PCO2. '
 'Urine output should be measured with an indwelling catheter. Fluid resuscitation with 0.9% saline is given until CVP '
 'reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg. Oliguria with hypotension is not a contraindication to '
 'vigorous fluid resuscitation, and the quantity of fluid required often exceeds normal blood volume. Dopamine may be '
 'given to increase mean BP to at least 60 mm Hg if a patient remains hypotensive after CVP or PAOP has been raised to '
 'target levels. If dopamine dose exceeds 20 μg/kg/min, another vasopressor such as norepinephrine may be added, but '
 'vasoconstriction caused by higher doses poses risks of organ hypoperfusion and acidosis. Oxygen is given by mask or '
 'nasal prongs, and tracheal intubation an


llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      55.01 ms /   316 runs   (    0.17 ms per token,  5743.99 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   30426.88 ms /   316 runs   (   96.29 ms per token,    10.39 tokens per second)
llama_print_timings:       total time =   31457.75 ms


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [133]:
for params in paramList:
  output(generate_rag_response(q2,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'


Llama.generate: prefix-match hit

llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      14.26 ms /   137 runs   (    0.10 ms per token,  9606.62 tokens per second)
llama_print_timings: prompt eval time =   12096.95 ms /  1868 tokens (    6.48 ms per token,   154.42 tokens per second)
llama_print_timings:        eval time =    8174.64 ms /   136 runs   (   60.11 ms per token,    16.64 tokens per second)
llama_print_timings:       total time =   20739.16 ms


('The context mentions that treatment for acute appendicitis is open or laparoscopic appendectomy. It also states that '
 'antibiotics are used in the treatment process, but they are not curative if the appendix is perforated. Therefore, '
 'surgery is necessary to remove the appendix and prevent complications such as gangrene, perforation, and '
 'peritonitis. The common symptoms for appendicitis include pain in the lower right abdomen, loss of appetite, nausea '
 'or vomiting, fever, and a feeling of being sick. So, no, appendicitis cannot be cured via medicine alone.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=1 tp=0.85 tk=100'


Llama.generate: prefix-match hit

llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      62.11 ms /   137 runs   (    0.45 ms per token,  2205.80 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   13934.26 ms /   137 runs   (  101.71 ms per token,     9.83 tokens per second)
llama_print_timings:       total time =   14450.74 ms


("The context mentions that treatment for acute appendicitis is through open or laparoscopic appendectomy. There's no "
 'mention of the disease being curable with medication alone. Symptoms for appendicitis include difficulty in locating '
 'the appendix, it usually lies behind the cecum or the ileum and mesentery of the right colon. Other symptoms are '
 "perforation, inflammation around the appendix preventing procedures, and a contraindication if there's inflammatory "
 'bowel disease involving the cecum. Therefore, appendicitis cannot be cured with medicine and requires surgical '
 'removal.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=2 tp=0.75 tk=250'


...


llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      18.58 ms /   113 runs   (    0.16 ms per token,  6082.46 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   13036.10 ms /   113 runs   (  115.36 ms per token,     8.67 tokens per second)
llama_print_timings:       total time =   13422.62 ms


'>>>>>>>>>Calling llm with mt=1000 temp=4 tp=0.45 tk=500'


Llama.generate: prefix-match hit


('The context mentions that treatment for acute appendicitis is open or laparoscopic appendectomy. It also states that '
 'antibiotics are used in the treatment process, but they are not curative on their own. Therefore, the answer would '
 'be:\n'
 '\n'
 'Appendicitis is typically characterized by symptoms such as abdominal pain, usually localized in the lower right '
 'quadrant of the abdomen, loss of appetite, nausea, and vomiting. These symptoms can worsen over time if left '
 'untreated. Appendicitis cannot be cured via medicine alone; surgical removal of the appendix (appendectomy) is '
 'required to treat the condition.')
'                                      ---------------------------'



llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      29.65 ms /   144 runs   (    0.21 ms per token,  4856.66 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   15391.96 ms /   144 runs   (  106.89 ms per token,     9.36 tokens per second)
llama_print_timings:       total time =   15863.65 ms


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [136]:
for params in paramList:
  output(generate_rag_response(q3,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'


Llama.generate: prefix-match hit

llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      29.84 ms /   328 runs   (    0.09 ms per token, 10993.06 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   29780.51 ms /   328 runs   (   90.79 ms per token,    11.01 tokens per second)
llama_print_timings:       total time =   31444.17 ms


('Based on the context provided, the condition being described is Alopecia Areata. The context mentions that it is a '
 'sudden patchy hair loss in people with no obvious skin or systemic disorder, affecting the scalp and beard most '
 'frequently, but any hairy area may be involved. It is thought to be an autoimmune disorder affecting genetically '
 'susceptible people exposed to unclear environmental triggers.\n'
 '\n'
 'The context also mentions that microscopic hair examination or scalp biopsy may be required for definitive '
 'diagnosis. As for treatments, the context suggests that corticosteroids, either topical or oral, are mentioned as '
 'potential options for treating Lichen planopilaris and chronic cutaneous lupus lesions, which could also be possible '
 'causes of patchy hair loss. However, the context does not specifically mention Alopecia Areata as being treated with '
 'corticosteroids. Other treatments mentioned for these conditions include oral antimalarials, retinoids

Llama.generate: prefix-match hit

llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      53.10 ms /   122 runs   (    0.44 ms per token,  2297.55 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   17849.78 ms /   122 runs   (  146.31 ms per token,     6.83 tokens per second)
llama_print_timings:       total time =   18343.41 ms


('Based on the context provided, the effective treatments for sudden patchy hair loss, also known as alopecia areata, '
 'include corticosteroids, oral antimalarials, retinoids, or immunosuppressants. The cause of this condition is '
 'thought to be an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental '
 'triggers. It can affect any hairy area, but the scalp and beard are most commonly affected. In some cases, it may '
 'progress to involve most or all of the body (alopecia universalis).')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=2 tp=0.75 tk=250'


...

Llama.generate: prefix-match hit

llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      25.98 ms /   153 runs   (    0.17 ms per token,  5890.28 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   16406.33 ms /   153 runs   (  107.23 ms per token,     9.33 tokens per second)
llama_print_timings:       total time =   16984.89 ms


('Based on the context provided, sudden patchy hair loss can be a symptom of alopecia areata. The context mentions '
 'that alopecia areata is an autoimmune disorder affecting genetically susceptible people exposed to unclear '
 'environmental triggers. The effective treatments or solutions for addressing this condition include corticosteroids, '
 'oral antimalarials, retinoids, and immunosuppressants. The context also mentions that a thorough evaluation should '
 'be conducted if concomitant virilization or scarring hair loss is present to identify the underlying disorder. '
 'Therefore, the possible causes behind sudden patchy hair loss could be an autoimmune disorder or other underlying '
 'conditions that require further investigation and treatment.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=4 tp=0.45 tk=500'


Llama.generate: prefix-match hit


('Based on the context provided, the condition being described is Alopecia Areata. The context mentions that it is a '
 'sudden patchy hair loss in people with no obvious skin or systemic disorder, and that it is thought to be an '
 'autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers.\n'
 '\n'
 'The context also provides some information on the treatments for Alopecia Areata. These include corticosteroids, '
 'oral antimalarials, retinoids, or immunosuppressants. However, it is important to note that the effectiveness of '
 'these treatments may vary from person to person and that a thorough evaluation by a healthcare professional is '
 'necessary for an accurate diagnosis and treatment plan.\n'
 '\n'
 'Therefore, the answer to the user question would be: The effective treatments for addressing sudden patchy hair '
 'loss, commonly seen as localized bald spots on the scalp, include corticosteroids, oral antimalarials, retinoids, or '



llama_print_timings:        load time =    3480.05 ms
llama_print_timings:      sample time =      51.55 ms /   250 runs   (    0.21 ms per token,  4849.47 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   20108.17 ms /   250 runs   (   80.43 ms per token,    12.43 tokens per second)
llama_print_timings:       total time =   21059.89 ms


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
for params in paramList:
  output(generate_rag_response(q4,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'


Llama.generate: prefix-match hit


('The context states that initial treatment for a traumatic brain injury (TBI) includes ensuring a reliable airway and '
 'maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more '
 'severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial '
 'pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining '
 'adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. '
 'Subsequently, many patients require rehabilitation. Therefore, the recommended treatments for a person who has '
 'sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, '
 'include initial medical treatment such as ensuring a reliable airway, maintaining adequate ventilation, oxygenation, '
 'and blood pressure, surgery if necessary, and rehabilitation afterwards


llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      17.74 ms /   188 runs   (    0.09 ms per token, 10595.73 tokens per second)
llama_print_timings: prompt eval time =    9427.66 ms /  1759 tokens (    5.36 ms per token,   186.58 tokens per second)
llama_print_timings:        eval time =   10372.51 ms /   187 runs   (   55.47 ms per token,    18.03 tokens per second)
llama_print_timings:       total time =   20342.52 ms


'>>>>>>>>>Calling llm with mt=1000 temp=1 tp=0.85 tk=100'


...

Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      26.79 ms /   160 runs   (    0.17 ms per token,  5971.93 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   12854.86 ms /   160 runs   (   80.34 ms per token,    12.45 tokens per second)
llama_print_timings:       total time =   13355.55 ms


('Based on the context provided, the initial treatment for a person who has sustained a physical injury to brain '
 'tissue includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. '
 'Surgery may be needed in patients with more severe injury to place monitors to track and treat intracranial '
 'pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first '
 'few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of '
 'altered sensorium are important. Subsequently, many patients require rehabilitation. The context also mentions that '
 'causes of traumatic brain injury (TBI) include motor vehicle crashes, falls, assaults, and sports activities.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=4 tp=0.45 tk=500'


Llama.generate: prefix-match hit


('I. Initial treatment includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and '
 'blood pressure. Surgery is often needed in patients with more severe injury to place monitors to track and treat '
 'intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. '
 'In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing '
 'complications of altered sensorium are important. Subsequently, many patients require rehabilitation.\n'
 '\n'
 'II. Structural changes from head injury may be gross or microscopic, depending on the mechanism and forces involved. '
 'Clinical manifestations vary markedly in severity and consequences. Injuries are commonly categorized as open or '
 'closed. Open injuries involve penetration of the scalp and skull (and usually the meninges and underlying brain '
 'tissue). They typically involve bullets or sharp objects, but a s


llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =     136.39 ms /   866 runs   (    0.16 ms per token,  6349.39 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   55161.92 ms /   866 runs   (   63.70 ms per token,    15.70 tokens per second)
llama_print_timings:       total time =   57607.58 ms


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
for params in paramList:
  output(generate_rag_response(q5,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'


Llama.generate: prefix-match hit


('Based on the context provided, the necessary precautions and treatment steps for a person who has fractured their '
 'leg during a hiking trip include:\n'
 '1. Sterile wound dressings for suspected open fractures.\n'
 '2. Tetanus prophylaxis.\n'
 '3. Broad-spectrum antibiotics (eg, a 2nd-generation cephalosporin plus an aminoglycoside).\n'
 '4. Use of a cane during recovery.\n'
 '\n'
 'The person should move the cane to the lower step shortly before descending with the bad leg for better support and '
 'stability. No further information about the care and recovery process is provided in the context.')
'                                      ---------------------------'



llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      14.47 ms /   146 runs   (    0.10 ms per token, 10089.14 tokens per second)
llama_print_timings: prompt eval time =    4699.63 ms /   517 tokens (    9.09 ms per token,   110.01 tokens per second)
llama_print_timings:        eval time =    7371.60 ms /   145 runs   (   50.84 ms per token,    19.67 tokens per second)
llama_print_timings:       total time =   12478.47 ms


'>>>>>>>>>Calling llm with mt=1000 temp=1 tp=0.85 tk=100'


...

Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      15.54 ms /    91 runs   (    0.17 ms per token,  5857.74 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =    7833.61 ms /    91 runs   (   86.08 ms per token,    11.62 tokens per second)
llama_print_timings:       total time =    8108.87 ms


("I don't know. The context does not provide information on the specific precautions and treatment steps for a person "
 'who has fractured their leg during a hiking trip. It only mentions that suspected open fractures require sterile '
 'wound dressings, tetanus prophylaxis, and broad-spectrum antibiotics, and that patients should use a cane when '
 'descending stairs with a bad leg during recovery.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=4 tp=0.45 tk=500'


Llama.generate: prefix-match hit


('Based on the context provided, the necessary precautions and treatment steps for a person who has fractured their '
 'leg during a hiking trip include:\n'
 '1. Sterile wound dressings for suspected open fractures.\n'
 '2. Tetanus prophylaxis.\n'
 '3. Broad-spectrum antibiotics (eg, a 2nd-generation cephalosporin plus an aminoglycoside).\n'
 '4. Use of a cane during recovery.\n'
 '\n'
 'The person should move the cane to the lower step shortly before descending with the bad leg for better support and '
 'stability. This information is mentioned in Chapter 350. Rehabilitation, but it is also relevant to the care and '
 'recovery of someone with a fractured leg.')
'                                      ---------------------------'



llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      40.57 ms /   164 runs   (    0.25 ms per token,  4042.30 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   12729.03 ms /   164 runs   (   77.62 ms per token,    12.88 tokens per second)
llama_print_timings:       total time =   13186.40 ms


### Comments / Observations - RAG
*   Adding RAG makes the responses refer specifically to the Merck Manual more often, but not every time depending on the parameters.  
*   Overall, the second or third of five parameter sets provides a good balance between wordiness and the amount of relevant information provided.
*   The responses are also more focused on first aid and acute medical intervention, and less focused on what the patient should do after receiving medical care.
* Adding RAG makes the queries run even longer, but only to a small extent.



## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [ ]:
groundedness_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
"""

In [ ]:
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluaton criteria and assign a score.
"""

In [ ]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [ ]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = get_retreiver().get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    output( ">>>>>>>>>Calling llm with mt=" + str(max_tokens) + " temp=" + str(temperature) + " tp=" + str(top_p) + " tk=" + str(top_k) + " and k=" + str(k) )
    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

* Note - max_tokens of 2048 being used for this section for all Q1-Q5

In [ ]:
#Checking groundedness and relevance for the middle parameter set and k=2, k=8
params = paramList[2]
output(params)
for k in [2,8]:
  ground, rel = generate_ground_relevance_response(q1, k=k, max_tokens=2048, temperature=params["temp"], top_p=params["top_p"], top_k=params["top_k"])
  output("Groundedness:")
  output(ground)
  output("Relevance:")
  output(rel)
  output("----------------------------------------------------")

{'max_tokens': 1000, 'temp': 2, 'top_k': 250, 'top_p': 0.75}
'>>>>>>>>>Calling llm with mt=2048 temp=2 tp=0.75 tk=250 and k=2'


Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      41.81 ms /   365 runs   (    0.11 ms per token,  8730.80 tokens per second)
llama_print_timings: prompt eval time =   15191.17 ms /  1285 tokens (   11.82 ms per token,    84.59 tokens per second)
llama_print_timings:        eval time =   18998.28 ms /   364 runs   (   52.19 ms per token,    19.16 tokens per second)
llama_print_timings:       total time =   35796.10 ms
Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      24.56 ms /   203 runs   (    0.12 ms per token,  8264.13 tokens per second)
llama_print_timings: prompt eval time =    6260.40 ms /  1758 tokens (    3.56 ms per token,   280.81 tokens per second)
llama_print_timings:        eval time =   10818.62 ms /   202 runs   (   53.56 ms per token,    18.67 tokens per second)
llama_print_timings:       total time =   17

'Groundedness:'
(' Steps to evaluate the answer:\n'
 '1. Identify the key information from the context that relates to managing sepsis in a critical care unit.\n'
 '2. Check if the AI generated answer includes all the essential steps and procedures for managing sepsis in a '
 'critical care unit as mentioned in the context.\n'
 '3. Determine if the AI generated answer is derived only from the information presented in the context and not any '
 'external sources or personal assumptions.\n'
 '\n'
 'Explanation:\n'
 'The AI generated answer adheres to the metric since it includes all the essential steps and procedures for managing '
 'sepsis in a critical care unit as mentioned in the context. The answer also does not include any external sources or '
 'personal assumptions, but rather solely derives the information from the context.\n'
 '\n'
 'Evaluation:\n'
 'The metric is followed completely.\n'
 '\n'
 'Rating:\n'
 'Based on the evaluation criteria and the complete adherence to the met

Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      30.36 ms /   212 runs   (    0.14 ms per token,  6982.18 tokens per second)
llama_print_timings: prompt eval time =   17638.36 ms /  3779 tokens (    4.67 ms per token,   214.25 tokens per second)
llama_print_timings:        eval time =   13268.10 ms /   211 runs   (   62.88 ms per token,    15.90 tokens per second)
llama_print_timings:       total time =   31654.16 ms
Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      28.08 ms /   236 runs   (    0.12 ms per token,  8405.16 tokens per second)
llama_print_timings: prompt eval time =   16162.99 ms /  4099 tokens (    3.94 ms per token,   253.60 tokens per second)
llama_print_timings:        eval time =   15609.52 ms /   235 runs   (   66.42 ms per token,    15.05 tokens per second)
llama_print_timings:       total time =   32

'Groundedness:'
(' To evaluate the answer with respect to the given metric, we need to follow these steps:\n'
 '\n'
 '1. Identify the information presented in the context that could potentially influence the generation of the answer. '
 'In this case, the context provides details about managing sepsis in a critical care unit, monitoring parameters, and '
 'recommended treatments including antibiotics and normalization of blood glucose levels.\n'
 '\n'
 '2. Check if the AI generated answer is derived solely from the information provided in the context without '
 'introducing any new or extraneous information. The answer provided mentions all the aspects discussed in the '
 'context, including vital signs monitoring, fluid intake and output measurement, blood tests, empiric antibiotic '
 'therapy with specific medications, normalization of blood glucose levels, and the continuation of antibiotics for a '
 'certain period after shock resolution. These are all derived from the context with


llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      44.53 ms /   348 runs   (    0.13 ms per token,  7815.31 tokens per second)
llama_print_timings: prompt eval time =   17859.52 ms /  3975 tokens (    4.49 ms per token,   222.57 tokens per second)
llama_print_timings:        eval time =   21928.14 ms /   347 runs   (   63.19 ms per token,    15.82 tokens per second)
llama_print_timings:       total time =   40624.22 ms


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
#Checking groundedness and relevance for the last parameter set and k=2, k=8
params = paramList[4]
output(params)
for k in [2,8]:
  ground, rel = generate_ground_relevance_response(q2, k=k, max_tokens=2048, temperature=params["temp"], top_p=params["top_p"], top_k=params["top_k"])
  output("Groundedness:")
  output(ground)
  output("Relevance:")
  output(rel)
  output("----------------------------------------------------")

{'max_tokens': 1000, 'temp': 4, 'top_k': 500, 'top_p': 0.45}
'>>>>>>>>>Calling llm with mt=2048 temp=4 tp=0.45 tk=500 and k=2'


Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      15.50 ms /    83 runs   (    0.19 ms per token,  5355.18 tokens per second)
llama_print_timings: prompt eval time =    9887.47 ms /  1426 tokens (    6.93 ms per token,   144.22 tokens per second)
llama_print_timings:        eval time =    4346.17 ms /    82 runs   (   53.00 ms per token,    18.87 tokens per second)
llama_print_timings:       total time =   14577.74 ms
Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      41.26 ms /   240 runs   (    0.17 ms per token,  5816.35 tokens per second)
llama_print_timings: prompt eval time =    5921.78 ms /  1617 tokens (    3.66 ms per token,   273.06 tokens per second)
llama_print_timings:        eval time =   13510.50 ms /   239 runs   (   56.53 ms per token,    17.69 tokens per second)
llama_print_timings:       total time =   20

'Groundedness:'
(' Steps to evaluate the answer:\n'
 '1. Identify the key information in the context related to appendicitis and its treatment.\n'
 '2. Check if the symptoms mentioned in the answer are consistent with those described for appendicitis in the '
 'context.\n'
 '3. Determine if the statement about the inability to cure appendicitis with medicine alone is accurate based on the '
 'context.\n'
 '4. Verify if the recommended treatment, which is an appendectomy, is consistent with the information provided in the '
 'context.\n'
 '\n'
 'Explanation:\n'
 'The answer adheres to the metric as it derives the information only from the context. The symptoms mentioned for '
 'appendicitis in the answer are consistent with those described in the context. The statement about the inability to '
 'cure appendicitis with medicine alone is accurate based on the context, and the recommended treatment of an '
 'appendectomy is consistent with the information provided in the context.\n'
 '\n'



llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      45.57 ms /   268 runs   (    0.17 ms per token,  5880.80 tokens per second)
llama_print_timings: prompt eval time =    5498.17 ms /  1493 tokens (    3.68 ms per token,   271.54 tokens per second)
llama_print_timings:        eval time =   14356.70 ms /   267 runs   (   53.77 ms per token,    18.60 tokens per second)
llama_print_timings:       total time =   20462.90 ms


'>>>>>>>>>Calling llm with mt=2048 temp=4 tp=0.45 tk=500 and k=8'


Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      31.04 ms /   194 runs   (    0.16 ms per token,  6251.01 tokens per second)
llama_print_timings: prompt eval time =   20630.64 ms /  4345 tokens (    4.75 ms per token,   210.61 tokens per second)
llama_print_timings:        eval time =   12181.63 ms /   193 runs   (   63.12 ms per token,    15.84 tokens per second)
llama_print_timings:       total time =   33417.30 ms
Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      57.06 ms /   346 runs   (    0.16 ms per token,  6063.47 tokens per second)
llama_print_timings: prompt eval time =   18053.90 ms /  4647 tokens (    3.89 ms per token,   257.40 tokens per second)
llama_print_timings:        eval time =   22430.22 ms /   345 runs   (   65.02 ms per token,    15.38 tokens per second)
llama_print_timings:       total time =   41

'Groundedness:'
(' To evaluate the answer according to the given metric, we need to follow these steps:\n'
 '\n'
 '1. Identify the information in the context related to the question about appendicitis symptoms and treatment.\n'
 '2. Determine if the AI-generated answer is derived only from the identified information in the context.\n'
 '\n'
 "First, let's identify the relevant information in the context regarding appendicitis symptoms and treatment:\n"
 '\n'
 '- The classic symptoms of acute appendicitis are epigastric or periumbilical pain followed by brief nausea, '
 'vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough '
 "and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point. "
 'Additional signs include pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing '
 'sign), an increase in pain from passive extension of the right hip joint 


llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      37.99 ms /   225 runs   (    0.17 ms per token,  5922.77 tokens per second)
llama_print_timings: prompt eval time =   17635.11 ms /  4523 tokens (    3.90 ms per token,   256.48 tokens per second)
llama_print_timings:        eval time =   14374.08 ms /   224 runs   (   64.17 ms per token,    15.58 tokens per second)
llama_print_timings:       total time =   32465.24 ms


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
#Checking groundedness and relevance for the first parameter set and k=2, k=8
params = paramList[0]
output(params)
for k in [2,8]:
  ground, rel = generate_ground_relevance_response(q3, k=k, max_tokens=2048, temperature=params["temp"], top_p=params["top_p"], top_k=params["top_k"])
  output("Groundedness:")
  output(ground)
  output("Relevance:")
  output(rel)
  output("----------------------------------------------------")

{'max_tokens': 1000, 'temp': 0, 'top_k': 50, 'top_p': 0.95}
'>>>>>>>>>Calling llm with mt=2048 temp=0 tp=0.95 tk=50 and k=2'


Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      12.07 ms /   127 runs   (    0.10 ms per token, 10524.57 tokens per second)
llama_print_timings: prompt eval time =    6809.78 ms /  1015 tokens (    6.71 ms per token,   149.05 tokens per second)
llama_print_timings:        eval time =    6577.62 ms /   126 runs   (   52.20 ms per token,    19.16 tokens per second)
llama_print_timings:       total time =   13790.81 ms
Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      23.92 ms /   245 runs   (    0.10 ms per token, 10242.47 tokens per second)
llama_print_timings: prompt eval time =    4501.99 ms /  1250 tokens (    3.60 ms per token,   277.66 tokens per second)
llama_print_timings:        eval time =   12639.11 ms /   244 runs   (   51.80 ms per token,    19.31 tokens per second)
llama_print_timings:       total time =   17

'Groundedness:'
(' Steps to evaluate the answer:\n'
 '1. Identify the main question and the specific information being asked for in the question.\n'
 '2. Read through the context carefully to understand the information provided about the topic.\n'
 '3. Determine if the AI generated answer is derived only from the information presented in the context.\n'
 '\n'
 'Explanation:\n'
 'The main question asks about effective treatments or solutions for sudden patchy hair loss and possible causes '
 'behind it. The context provides information on various conditions that can cause hair loss, their treatments, and '
 'alopecia areata as a specific condition. The AI generated answer correctly identifies alopecia areata as a condition '
 'causing sudden patchy hair loss and mentions corticosteroids as one of the treatments for it. It also mentions other '
 'treatments for different conditions mentioned in the context. Therefore, the answer is derived from the information '
 'presented in the contex

Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      15.25 ms /   163 runs   (    0.09 ms per token, 10688.52 tokens per second)
llama_print_timings: prompt eval time =   20782.92 ms /  4568 tokens (    4.55 ms per token,   219.80 tokens per second)
llama_print_timings:        eval time =   10365.93 ms /   162 runs   (   63.99 ms per token,    15.63 tokens per second)
llama_print_timings:       total time =   31688.73 ms
Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      13.74 ms /   143 runs   (    0.10 ms per token, 10405.30 tokens per second)
llama_print_timings: prompt eval time =   18925.01 ms /  4839 tokens (    3.91 ms per token,   255.69 tokens per second)
llama_print_timings:        eval time =    9377.15 ms /   142 runs   (   66.04 ms per token,    15.14 tokens per second)
llama_print_timings:       total time =   28

'Groundedness:'
(' Steps to evaluate the answer:\n'
 '1. Identify the main topic of the question and context.\n'
 '2. Determine if the answer is derived only from the information presented in the context.\n'
 '3. Check if the answer mentions any additional or incorrect information not present in the context.\n'
 '\n'
 'Evaluation:\n'
 'The answer adheres to the metric as it mentions the treatment options for sudden patchy hair loss (alopecia areata) '
 'which are explicitly stated in the context. The answer does not include any additional or incorrect information.\n'
 '\n'
 'Rating:\n'
 'Based on the given evaluation criteria, I would rate the answer as a 5 since the metric is followed completely.')
'Relevance:'
(' Steps to evaluate the context as per the metric:\n'
 '1. Identify the main aspects of the question: effective treatments or solutions for addressing sudden patchy hair '
 'loss and possible causes behind it.\n'
 '2. Determine if all important aspects are contained in the con


llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      12.25 ms /   127 runs   (    0.10 ms per token, 10366.50 tokens per second)
llama_print_timings: prompt eval time =   18516.50 ms /  4715 tokens (    3.93 ms per token,   254.64 tokens per second)
llama_print_timings:        eval time =    8364.07 ms /   126 runs   (   66.38 ms per token,    15.06 tokens per second)
llama_print_timings:       total time =   27149.58 ms


### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
#Checking groundedness and relevance for the second parameter set and k=3, k=5
params = paramList[1]
output(params)
for k in [3,5]:
  ground, rel = generate_ground_relevance_response(q4, k=k, max_tokens=2048, temperature=params["temp"], top_p=params["top_p"], top_k=params["top_k"])
  output("Groundedness:")
  output(ground)
  output("Relevance:")
  output(rel)
  output("----------------------------------------------------")

{'max_tokens': 1000, 'temp': 1, 'top_k': 100, 'top_p': 0.85}
'>>>>>>>>>Calling llm with mt=2048 temp=1 tp=0.85 tk=100 and k=3'


Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      13.72 ms /   146 runs   (    0.09 ms per token, 10643.73 tokens per second)
llama_print_timings: prompt eval time =    9693.22 ms /  1925 tokens (    5.04 ms per token,   198.59 tokens per second)
llama_print_timings:        eval time =    7962.05 ms /   145 runs   (   54.91 ms per token,    18.21 tokens per second)
llama_print_timings:       total time =   18094.17 ms
Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      16.86 ms /   189 runs   (    0.09 ms per token, 11210.63 tokens per second)
llama_print_timings: prompt eval time =    8063.04 ms /  2179 tokens (    3.70 ms per token,   270.25 tokens per second)
llama_print_timings:        eval time =   10517.81 ms /   188 runs   (   55.95 ms per token,    17.87 tokens per second)
llama_print_timings:       total time =   19

'Groundedness:'
(' Steps for evaluating the answer:\n'
 '1. Identify the information relevant to answering the question in the context.\n'
 '2. Determine if the answer is derived solely from the information presented in the context.\n'
 '\n'
 'Evaluation:\n'
 'The answer follows the metric to a good extent as it primarily focuses on treatments mentioned in the context, which '
 'include maintaining adequate brain perfusion and oxygenation, preventing complications of altered sensorium, surgery '
 'for more severe injuries, and rehabilitation. However, the answer also includes some additional information about '
 'the importance of family education during rehabilitation, which is not explicitly stated in the context but can be '
 'inferred from it. Thus, the answer slightly deviates from the metric by incorporating related information beyond '
 "what's presented directly within the context.\n"
 '\n'
 'Rating:\n'
 'Based on the evaluation, I would rate the answer a 3 (followed to a good 


llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      22.31 ms /   232 runs   (    0.10 ms per token, 10400.79 tokens per second)
llama_print_timings: prompt eval time =    7651.86 ms /  2055 tokens (    3.72 ms per token,   268.56 tokens per second)
llama_print_timings:        eval time =   12995.59 ms /   231 runs   (   56.26 ms per token,    17.78 tokens per second)
llama_print_timings:       total time =   21149.69 ms


'>>>>>>>>>Calling llm with mt=2048 temp=1 tp=0.85 tk=100 and k=5'


Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      11.08 ms /   109 runs   (    0.10 ms per token,  9837.55 tokens per second)
llama_print_timings: prompt eval time =   12258.24 ms /  2731 tokens (    4.49 ms per token,   222.79 tokens per second)
llama_print_timings:        eval time =    6329.31 ms /   108 runs   (   58.60 ms per token,    17.06 tokens per second)
llama_print_timings:       total time =   18849.10 ms
Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      21.18 ms /   210 runs   (    0.10 ms per token,  9917.36 tokens per second)
llama_print_timings: prompt eval time =   11128.66 ms /  2948 tokens (    3.77 ms per token,   264.90 tokens per second)
llama_print_timings:        eval time =   12394.27 ms /   209 runs   (   59.30 ms per token,    16.86 tokens per second)
llama_print_timings:       total time =   23

'Groundedness:'
(' Steps to evaluate the answer:\n'
 '1. Identify the key information in the context regarding treatments for a person with a traumatic brain injury '
 '(TBI).\n'
 '2. Compare this information with the AI generated answer to ensure that the answer is derived only from the context '
 'and not any external sources.\n'
 '\n'
 'Evaluation of adherence to metric:\n'
 'The AI generated answer adheres mostly to the metric as it mentions the treatments mentioned in the context, but '
 'also includes additional information about the importance of maintaining brain perfusion and oxygenation and '
 'preventing complications which is not explicitly stated in the context, but can be inferred from it.\n'
 '\n'
 'Rating based on evaluation criteria:\n'
 'Based on the extent to which the metric is followed, I would rate the answer as a 4. The AI generated answer follows '
 'the metric mostly, but there are some minor additional details that are not directly taken from the context.\n'
 


llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      41.30 ms /   423 runs   (    0.10 ms per token, 10243.37 tokens per second)
llama_print_timings: prompt eval time =   10676.79 ms /  2824 tokens (    3.78 ms per token,   264.50 tokens per second)
llama_print_timings:        eval time =   25126.90 ms /   422 runs   (   59.54 ms per token,    16.79 tokens per second)
llama_print_timings:       total time =   36591.12 ms


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
#Checking groundedness and relevance for the second parameter set and k=2, k=3
params = paramList[1]
output(params)
for k in [2,3]:
  ground, rel = generate_ground_relevance_response(q5, k=k, max_tokens=2048, temperature=params["temp"], top_p=params["top_p"], top_k=params["top_k"])
  output("Groundedness:")
  output(ground)
  output("Relevance:")
  output(rel)
  output("----------------------------------------------------")

{'max_tokens': 1000, 'temp': 1, 'top_k': 100, 'top_p': 0.85}
'>>>>>>>>>Calling llm with mt=2048 temp=1 tp=0.85 tk=100 and k=2'


Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =       9.78 ms /    99 runs   (    0.10 ms per token, 10121.66 tokens per second)
llama_print_timings: prompt eval time =    6497.27 ms /   564 tokens (   11.52 ms per token,    86.81 tokens per second)
llama_print_timings:        eval time =    5021.54 ms /    98 runs   (   51.24 ms per token,    19.52 tokens per second)
llama_print_timings:       total time =   11883.78 ms
Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      23.81 ms /   233 runs   (    0.10 ms per token,  9787.45 tokens per second)
llama_print_timings: prompt eval time =    2807.54 ms /   771 tokens (    3.64 ms per token,   274.62 tokens per second)
llama_print_timings:        eval time =   11860.20 ms /   232 runs   (   51.12 ms per token,    19.56 tokens per second)
llama_print_timings:       total time =   15

'Groundedness:'
(' Steps for evaluation:\n'
 '1. Identify the information related to the question (precautions and treatment) in the context.\n'
 '2. Determine if the AI generated answer includes only the information from the context.\n'
 '3. Evaluate the extent to which the answer follows the metric.\n'
 '\n'
 'Explanation:\n'
 'The AI generated answer is derived solely from the context, as it mentions the same precautions and treatment for a '
 'person with a suspected open fracture of their leg, which is stated in the context. Therefore, the answer follows '
 'the metric to a good extent (score 3).\n'
 '\n'
 "However, it's important to note that the context does not provide information about care and recovery steps beyond "
 'the mentioned precautions and treatment. The AI generated answer only reflects this limited information from the '
 'context, so it may not be comprehensive enough for someone seeking a full understanding of the topic. Nevertheless, '
 'based on the provided me

Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =       9.21 ms /    95 runs   (    0.10 ms per token, 10316.00 tokens per second)
llama_print_timings: prompt eval time =    7298.06 ms /   683 tokens (   10.69 ms per token,    93.59 tokens per second)
llama_print_timings:        eval time =    4784.91 ms /    94 runs   (   50.90 ms per token,    19.65 tokens per second)
llama_print_timings:       total time =   12437.54 ms
Llama.generate: prefix-match hit

llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      21.08 ms /   233 runs   (    0.09 ms per token, 11054.70 tokens per second)
llama_print_timings: prompt eval time =    3157.92 ms /   886 tokens (    3.56 ms per token,   280.56 tokens per second)
llama_print_timings:        eval time =   11986.50 ms /   232 runs   (   51.67 ms per token,    19.36 tokens per second)
llama_print_timings:       total time =   15

'Groundedness:'
(' Steps to evaluate the answer:\n'
 '1. Identify the key information in the context that is relevant to the question.\n'
 '2. Determine if the AI generated answer includes all the necessary precautions and treatment steps mentioned in the '
 'context for a person with a fractured leg.\n'
 '3. Verify if the information in the answer is derived only from the context and not from any external sources or '
 'assumptions.\n'
 '\n'
 'Explanation:\n'
 'The AI generated answer adheres to the metric as it includes all the necessary precautions (use of sterile wound '
 'dressings, tetanus prophylaxis, and broad-spectrum antibiotics) and treatment steps (using a cane) mentioned in the '
 'context for a person with a fractured leg. Additionally, the information in the answer is derived only from the '
 'context and not from any external sources or assumptions.\n'
 '\n'
 'Evaluation:\n'
 'The metric is followed completely as the answer includes all the necessary precautions and tre


llama_print_timings:        load time =    3001.69 ms
llama_print_timings:      sample time =      28.89 ms /   306 runs   (    0.09 ms per token, 10590.43 tokens per second)
llama_print_timings: prompt eval time =    2742.57 ms /   762 tokens (    3.60 ms per token,   277.84 tokens per second)
llama_print_timings:        eval time =   15835.98 ms /   305 runs   (   51.92 ms per token,    19.26 tokens per second)
llama_print_timings:       total time =   19135.55 ms


## Actionable Insights and Business Recommendations

- Higher values of retreival parameter "k" slightly degrades the groundedness and relevance of LLM responses using RAG.
- Higher values of max_tokens are needed for quality responses when providing context and espacially with RAG.  Otherwise, the responses get truncated and are not helpful.
- Even higher values of max_tokens are needed when evaluating the groundedness and relevance of LLM responses use RAG.
- Vector database creation time is a long running process that increases with PDF document size.  It would be ideal to create a set of these with different configurations up front and then evaluation which works best for a specific use case.


<font size=6 color='blue'>Power Ahead</font>
___